# Finished REBUS Run Comparison

Local, finished-run analysis for the REBUS-family Modal runs.

This notebook deliberately avoids live Modal volume calls. It reads downloaded artifacts under `logs/modal/` and auto-detects finished runs as soon as their logs are present.

Primary questions:
- How do the REBUS variants compare on headline evaluation metrics?
- Where do belief-confidence traces degrade by seed, step, context, or variant?
- Do reflective diagnostics show over-intervention, under-intervention, or better-targeted repair?
- When the hybrid repair-mask logs arrive, do they isolate useful repair from broad caution?

## 1. Imports and Configuration

In [ ]:

from pathlib import Path
import json
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# Path handling:
# If the notebook is launched from the repo or repo/notebooks, this auto-detects the repo root.
# If your Jupyter kernel runs from somewhere else, edit REPO_ROOT_OVERRIDE below.
REPO_ROOT_OVERRIDE = None  # e.g. Path('/Users/stephenbeale/Projects/ToM_AI_Research_Team')
HARDCODED_REPO_ROOT = Path('/Users/stephenbeale/Projects/ToM_AI_Research_Team')

candidate_roots = []
if REPO_ROOT_OVERRIDE is not None:
    candidate_roots.append(Path(REPO_ROOT_OVERRIDE).expanduser())
for candidate in [Path.cwd(), Path.cwd().parent, HARDCODED_REPO_ROOT]:
    if candidate not in candidate_roots:
        candidate_roots.append(candidate)

REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'logs' / 'modal').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    # Keep a concrete fallback so the diagnostic printout is actionable.
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser() if REPO_ROOT_OVERRIDE is not None else HARDCODED_REPO_ROOT

LOG_ROOT = REPO_ROOT / 'logs' / 'modal'
TARGET_TOTAL = 140000
SEEDS = [7, 11, 17, 23, 29]

# Expected locations. Missing folders are fine; they will appear once logs are downloaded.
RUN_SPECS = [
    ('auxhead_clear_reference', 'auxhead-clear reference', LOG_ROOT / 'auxhead-clear-20260416' / 'by-seed'),
    ('rebus_prospective', 'REBUS prospective', LOG_ROOT / 'auxhead-clear-rebus-20260621' / 'auxhead-clear-rebus-20260621'),
    ('rebus_reflective', 'REBUS reflective', LOG_ROOT / 'auxhead-clear-rebus-reflective-20260621' / 'auxhead-clear-rebus-reflective-20260621'),
    ('rebus_post_error_alpha_only', 'REBUS post-error alpha-only', LOG_ROOT / 'auxhead-clear-rebus-reflective-post-error-alpha-only-20260621' / 'auxhead-clear-rebus-reflective-post-error-alpha-only-20260621'),
    ('rebus_hybrid_repair_mask', 'REBUS hybrid repair-mask', LOG_ROOT / 'auxhead-clear-rebus-reflective-hybrid-repair-mask-20260621' / 'auxhead-clear-rebus-reflective-hybrid-repair-mask-20260621'),
    ('rebus_hybrid_explicit_mask', 'REBUS hybrid explicit-mask', LOG_ROOT / 'auxhead-clear-rebus-reflective-hybrid-explicit-mask-20260621' / 'auxhead-clear-rebus-reflective-hybrid-explicit-mask-20260621'),
    ('rebus_explicit_resolution', 'REBUS explicit resolution', LOG_ROOT / 'auxhead-clear-rebus-explicit-resolution-20260621' / 'auxhead-clear-rebus-explicit-resolution-20260621'),
]

EXPECTED_RUN_ROOTS = {key: root for key, _, root in RUN_SPECS}
VARIANT_ORDER = [key for key, _, _ in RUN_SPECS]
VARIANT_LABELS = {key: label for key, label, _ in RUN_SPECS}

METRIC_COLUMNS = [
    'AmbiguityEfficiency',
    'AverageDelay',
    'CollisionRate',
    'CoordinationEfficiency',
    'DeadlockRate',
    'IntentionPredictionF1',
    'StrategySwitchAccuracy',
    'SuccessRate',
    'ToMCoordScore',
]

RUN_COLUMNS = [
    'variant', 'variant_label', 'seed', 'target_total_episodes', 'completed_total_episodes',
    'additional_train_episodes', 'returncode', 'state', 'run_dir', 'analysis_json',
] + METRIC_COLUMNS

STEP_COLUMNS = [
    'variant', 'variant_label', 'seed', 'scenario_idx', 'step', 'scenario_family', 'partner_style',
    'outcome', 'context_sensitive_action_regret', 'urgency', 'norm', 'margin', 'timeout_pressure',
    'action_label', 'belief_confidence', 'belief_entropy', 'belief_class', 'action_confidence_margin',
    'context', 'evidence_released', 'rebus_alpha', 'rebus_prediction_error', 'rebus_prospective_error',
    'rebus_reflective_error', 'rebus_repair_window', 'rebus_surprise', 'rebus_outcome_surprise',
    'rebus_explicit_error', 'rebus_resolution_signal',
]

SCENARIO_COLUMNS = [
    'variant', 'variant_label', 'seed', 'scenario_idx', 'scenario_family', 'partner_style',
    'context_tag', 'outcome', 'context_sensitive_action_regret', 'belief_confidence_peak',
    'belief_shift_moment', 'action_switch_moment', 'steps',
]

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
print('cwd=', Path.cwd())
print('repo_root=', REPO_ROOT, 'exists=', REPO_ROOT.exists())
print('log_root=', LOG_ROOT, 'exists=', LOG_ROOT.exists())
if LOG_ROOT.exists():
    print('log_root children:', sorted(x.name for x in LOG_ROOT.iterdir() if x.is_dir())[:40])
else:
    print('If log_root is false, set REPO_ROOT_OVERRIDE in this cell to the repo path visible to this Jupyter kernel.')


## 2. Load Downloaded Run Artifacts

In [ ]:

def _safe_load_json(path: Path):
    if path is None:
        return None
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return None


def _load_choice_analysis_from_stdout(run_dir: Path):
    stdout_path = run_dir / 'stdout.log'
    if not stdout_path.exists():
        return None
    for line in stdout_path.read_text(errors='replace').splitlines():
        if line.startswith('choice_context_analysis='):
            try:
                return json.loads(line.split('=', 1)[1])
            except json.JSONDecodeError:
                return None
    return None


def find_run_dirs(root: Path, target_total: int = TARGET_TOTAL):
    if not root.exists():
        return []
    dirs = []
    for summary_path in root.glob(f'**/target-{target_total}/run_summary.json'):
        run_dir = summary_path.parent
        if run_dir not in dirs:
            dirs.append(run_dir)
    return sorted(dirs, key=lambda p: (str(p)))


def find_analysis_json(run_dir: Path):
    analysis_files = sorted((run_dir / 'analysis').glob('choice-analysis-*.json'))
    return analysis_files[-1] if analysis_files else None


def load_all_runs():
    run_rows = []
    analysis_payloads = {}
    missing = []
    for variant, root in EXPECTED_RUN_ROOTS.items():
        run_dirs = find_run_dirs(root)
        if not run_dirs:
            missing.append((variant, root))
            continue
        for run_dir in run_dirs:
            summary = _safe_load_json(run_dir / 'run_summary.json') or {}
            status = _safe_load_json(run_dir / 'run_status.json') or {}
            progress = _safe_load_json(run_dir / 'progress.json') or {}
            seed = summary.get('seed') or status.get('seed') or progress.get('seed')
            analysis_path = find_analysis_json(run_dir)
            analysis = _safe_load_json(analysis_path) if analysis_path else _load_choice_analysis_from_stdout(run_dir)
            key = (variant, int(seed) if seed is not None else None)
            if analysis is not None:
                analysis_payloads[key] = analysis
            eval_metrics = summary.get('eval_metrics') or {}
            row = {
                'variant': variant,
                'variant_label': VARIANT_LABELS.get(variant, variant),
                'seed': int(seed) if seed is not None else None,
                'target_total_episodes': summary.get('target_total_episodes') or status.get('completed_total_episodes') or progress.get('completed_total_episodes'),
                'completed_total_episodes': status.get('completed_total_episodes') or progress.get('completed_total_episodes'),
                'additional_train_episodes': summary.get('additional_train_episodes') or progress.get('completed_additional_episodes'),
                'returncode': summary.get('returncode'),
                'state': status.get('state'),
                'run_dir': str(run_dir),
                'analysis_json': str(analysis_path) if analysis_path else ('stdout.log' if analysis is not None else None),
            }
            for metric in METRIC_COLUMNS:
                row[metric] = eval_metrics.get(metric)
            run_rows.append(row)
    run_df = pd.DataFrame(run_rows, columns=RUN_COLUMNS)
    if not run_df.empty:
        run_df['variant'] = pd.Categorical(run_df['variant'], categories=VARIANT_ORDER, ordered=True)
        run_df = run_df.sort_values(['variant', 'seed']).reset_index(drop=True)
    return run_df, analysis_payloads, missing

run_df, analysis_payloads, missing_roots = load_all_runs()
print('loaded_runs=', len(run_df), 'loaded_analysis_payloads=', len(analysis_payloads))
print('missing_or_not_yet_downloaded:')
for variant, root in missing_roots:
    print(' -', variant, root, '| exists=', root.exists())

if run_df.empty:
    print('\nNo runs found. Check Box 1 path diagnostics above. If LOG_ROOT exists but roots are missing, verify the downloaded folder names under logs/modal.')
else:
    display(run_df)


## 3. Headline Metric Tables

These tables match the screenshot metric columns where `eval_metrics` are present. The old auxhead-clear reference logs may have analysis JSON but not eval metrics in `run_summary.json`; that is expected.

In [ ]:
if run_df.empty or 'seed' not in run_df.columns:
    metric_df = pd.DataFrame(columns=RUN_COLUMNS)
    print('No loaded runs yet. Run Box 1 and Box 2 after checking LOG_ROOT, or set REPO_ROOT_OVERRIDE in Box 1.')
else:
    metric_df = run_df.dropna(subset=['seed']).copy()
    for col in METRIC_COLUMNS:
        metric_df[col] = pd.to_numeric(metric_df[col], errors='coerce')
    metric_df = metric_df.dropna(subset=METRIC_COLUMNS, how='all')

    if metric_df.empty:
        print('No eval_metrics found yet.')
    else:
        display(metric_df[['variant_label', 'seed'] + METRIC_COLUMNS])

        mean_df = metric_df.groupby('variant_label', observed=True)[METRIC_COLUMNS].mean().reindex(
            [VARIANT_LABELS[v] for v in VARIANT_ORDER if VARIANT_LABELS[v] in set(metric_df['variant_label'])]
        )
        sd_df = metric_df.groupby('variant_label', observed=True)[METRIC_COLUMNS].std().reindex(mean_df.index)
        print('Mean by variant')
        display(mean_df)
        print('SD by variant')
        display(sd_df)


## 4. Metric Deltas vs Prospective REBUS

The prospective REBUS run is the most useful reference for the new reflective variants. Deltas are variant mean minus prospective mean.

In [ ]:
if metric_df.empty or 'REBUS prospective' not in set(metric_df['variant_label']):
    print('Prospective REBUS metrics not available yet.')
else:
    mean_df = metric_df.groupby('variant_label', observed=True)[METRIC_COLUMNS].mean()
    reference = mean_df.loc['REBUS prospective']
    delta_df = mean_df.subtract(reference, axis=1)
    display(delta_df)

    key_metrics = ['ToMCoordScore', 'SuccessRate', 'CollisionRate', 'DeadlockRate', 'IntentionPredictionF1']
    available = [c for c in key_metrics if c in delta_df.columns]
    ax = delta_df[available].plot(kind='bar', figsize=(12, 5), rot=30)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title('Mean metric deltas vs REBUS prospective')
    ax.set_ylabel('delta')
    plt.tight_layout()
    plt.show()


## 4b. Outcome Tradeoff: Success vs Failure

This view focuses on the narrow claim: did a variant improve success without buying that gain through more collisions or deadlocks? The second chart flips collision/deadlock deltas so all bars above zero are favorable.


In [ ]:

required = {'variant_label', 'SuccessRate', 'CollisionRate', 'DeadlockRate'}
if metric_df.empty or not required.issubset(metric_df.columns):
    print('Outcome-rate metrics not available yet.')
else:
    outcome_df = metric_df.dropna(subset=['SuccessRate', 'CollisionRate', 'DeadlockRate'], how='any').copy()
    if outcome_df.empty:
        print('No complete success/collision/deadlock metrics available yet.')
    else:
        outcome_means = outcome_df.groupby('variant_label', observed=True)[['SuccessRate', 'CollisionRate', 'DeadlockRate']].mean()
        outcome_means['TimeoutOrOtherFailureRate'] = (
            1.0 - outcome_means['SuccessRate'] - outcome_means['CollisionRate'] - outcome_means['DeadlockRate']
        ).clip(lower=0.0)
        desired_order = [VARIANT_LABELS[v] for v in VARIANT_ORDER if VARIANT_LABELS[v] in outcome_means.index]
        outcome_means = outcome_means.reindex(desired_order)
        display(outcome_means)

        stacked_cols = ['SuccessRate', 'TimeoutOrOtherFailureRate', 'DeadlockRate', 'CollisionRate']
        stacked_labels = ['success', 'timeout/other failure', 'deadlock', 'collision']
        stacked_colors = ['#2ca02c', '#bdbdbd', '#ff7f0e', '#d62728']
        ax = outcome_means[stacked_cols].plot(
            kind='bar',
            stacked=True,
            figsize=(12, 5),
            color=stacked_colors,
            rot=30,
        )
        ax.set_title('Outcome composition by variant')
        ax.set_ylabel('mean rate')
        ax.set_ylim(0, 1.0)
        ax.legend(stacked_labels, loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()
        plt.show()

        if 'REBUS prospective' not in outcome_means.index:
            print('Prospective REBUS reference is unavailable for tradeoff deltas.')
        else:
            reference = outcome_means.loc['REBUS prospective']
            tradeoff = pd.DataFrame(index=outcome_means.index)
            tradeoff['success gain'] = outcome_means['SuccessRate'] - reference['SuccessRate']
            tradeoff['collision reduction'] = reference['CollisionRate'] - outcome_means['CollisionRate']
            tradeoff['deadlock reduction'] = reference['DeadlockRate'] - outcome_means['DeadlockRate']
            tradeoff['collision+deadlock reduction'] = (
                reference['CollisionRate'] + reference['DeadlockRate']
                - outcome_means['CollisionRate'] - outcome_means['DeadlockRate']
            )
            display(tradeoff)

            ax = tradeoff[['success gain', 'collision reduction', 'deadlock reduction']].plot(
                kind='bar',
                figsize=(12, 5),
                color=['#2ca02c', '#d62728', '#ff7f0e'],
                rot=30,
            )
            ax.axhline(0, color='black', linewidth=1)
            ax.set_title('Outcome tradeoff vs REBUS prospective (higher is better)')
            ax.set_ylabel('rate delta')
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()

            if 'REBUS hybrid explicit-mask' in tradeoff.index:
                explicit = tradeoff.loc['REBUS hybrid explicit-mask']
                print(
                    'Explicit-mask vs prospective: '
                    f"success {explicit['success gain']:+.3f}, "
                    f"collision {explicit['collision reduction']:+.3f} reduction, "
                    f"deadlock {explicit['deadlock reduction']:+.3f} reduction."
                )


## 5. Flatten Choice Analysis Traces

This creates one row per scenario step, including belief confidence, context, action, outcome, and REBUS diagnostics where present.

In [ ]:

def flatten_choice_analysis(analysis_payloads):
    rows = []
    scenario_rows = []
    for (variant, seed), payload in analysis_payloads.items():
        variant_label = VARIANT_LABELS.get(variant, variant)
        for scenario_idx, scenario in enumerate(payload.get('scenario_summaries', [])):
            action_trace = scenario.get('action_trace', []) or []
            n = len(action_trace)
            scenario_rows.append({
                'variant': variant,
                'variant_label': variant_label,
                'seed': seed,
                'scenario_idx': scenario_idx,
                'scenario_family': scenario.get('scenario_family'),
                'partner_style': scenario.get('partner_style'),
                'context_tag': scenario.get('context_tag'),
                'outcome': scenario.get('outcome'),
                'context_sensitive_action_regret': scenario.get('context_sensitive_action_regret'),
                'belief_confidence_peak': scenario.get('belief_confidence_peak'),
                'belief_shift_moment': scenario.get('belief_shift_moment'),
                'action_switch_moment': scenario.get('action_switch_moment'),
                'steps': n,
            })
            tag_set = scenario.get('context_tag_set') or {}
            def get_trace(name, default=np.nan):
                values = scenario.get(name, []) or []
                return values + [default] * max(0, n - len(values))
            traces = {
                'action_label': get_trace('action_label_trace', None),
                'belief_confidence': get_trace('belief_confidence_trace'),
                'belief_entropy': get_trace('belief_entropy_trace'),
                'belief_class': get_trace('belief_class_trace'),
                'action_confidence_margin': get_trace('action_confidence_margin_trace'),
                'context': get_trace('context_trace', None),
                'evidence_released': get_trace('evidence_released_trace'),
                'rebus_alpha': get_trace('rebus_alpha_trace'),
                'rebus_prediction_error': get_trace('rebus_prediction_error_trace'),
                'rebus_prospective_error': get_trace('rebus_prospective_error_trace'),
                'rebus_reflective_error': get_trace('rebus_reflective_error_trace'),
                'rebus_repair_window': get_trace('rebus_repair_window_trace'),
                'rebus_surprise': get_trace('rebus_surprise_trace'),
                'rebus_outcome_surprise': get_trace('rebus_outcome_surprise_trace'),
                'rebus_explicit_error': get_trace('rebus_explicit_error_trace'),
                'rebus_resolution_signal': get_trace('rebus_resolution_signal_trace'),
            }
            for i in range(n):
                row = {
                    'variant': variant,
                    'variant_label': variant_label,
                    'seed': seed,
                    'scenario_idx': scenario_idx,
                    'step': i + 1,
                    'scenario_family': scenario.get('scenario_family'),
                    'partner_style': scenario.get('partner_style'),
                    'outcome': scenario.get('outcome'),
                    'context_sensitive_action_regret': scenario.get('context_sensitive_action_regret'),
                    'urgency': tag_set.get('urgency'),
                    'norm': tag_set.get('norm'),
                    'margin': tag_set.get('margin'),
                    'timeout_pressure': tag_set.get('timeout_pressure'),
                }
                for name, values in traces.items():
                    row[name] = values[i] if i < len(values) else np.nan
                rows.append(row)
    step_df = pd.DataFrame(rows, columns=STEP_COLUMNS)
    scenario_df = pd.DataFrame(scenario_rows, columns=SCENARIO_COLUMNS)
    if not step_df.empty:
        step_df['variant'] = pd.Categorical(step_df['variant'], categories=VARIANT_ORDER, ordered=True)
        step_df = step_df.sort_values(['variant', 'seed', 'scenario_idx', 'step']).reset_index(drop=True)
    return step_df, scenario_df

step_df, scenario_df = flatten_choice_analysis(analysis_payloads)
print('step rows=', len(step_df), 'scenario rows=', len(scenario_df))
display(step_df.head())


## 6. Belief Confidence by Step

This reproduces the style of the enclosed degradation plot, but can compare variants and/or isolate one seed.

In [ ]:
def summarize_by_step(df, value_col='belief_confidence', group_cols=('variant_label', 'step')):
    use = df.dropna(subset=[value_col]).copy()
    if use.empty:
        return pd.DataFrame()
    grouped = use.groupby(list(group_cols), observed=True)[value_col]
    out = grouped.agg(['mean', 'std', 'count']).reset_index()
    out['se'] = out['std'] / np.sqrt(out['count'].clip(lower=1))
    return out


def plot_belief_confidence_by_step(seed=None, variants=None, include_reference=True, title_suffix=''):
    df = step_df.copy()
    if df.empty:
        print('No step traces loaded.')
        return
    if seed is not None:
        df = df[df['seed'] == seed]
    if variants is not None:
        labels = [VARIANT_LABELS.get(v, v) for v in variants]
        df = df[df['variant_label'].isin(labels)]
    if not include_reference:
        df = df[df['variant'] != 'auxhead_clear_reference']
    summary = summarize_by_step(df)
    if summary.empty:
        print('No belief-confidence traces for this selection.')
        return
    plt.figure(figsize=(12, 6))
    for label, sub in summary.groupby('variant_label', observed=True):
        sub = sub.sort_values('step')
        x = sub['step'].to_numpy(dtype=float)
        y = sub['mean'].to_numpy(dtype=float)
        se = sub['se'].fillna(0).to_numpy(dtype=float)
        plt.plot(x, y, marker='o', linewidth=2, label=label)
        plt.fill_between(x, y - se, y + se, alpha=0.15)
    plt.ylim(0, 1.02)
    plt.xlabel('step')
    plt.ylabel('mean belief confidence')
    seed_txt = f' | seed {seed}' if seed is not None else ' | all seeds'
    plt.title('Mean belief confidence by step' + seed_txt + title_suffix)
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_belief_confidence_by_step(seed=29)
plot_belief_confidence_by_step(seed=None, include_reference=False, title_suffix=' | REBUS variants only')


## 7. Belief-Confidence Degradation Metrics

Definitions:
- `peak_to_final_drop`: maximum mean confidence across steps minus final-step mean confidence.
- `late_drop`: mean over the last three available steps subtracted from the peak.
- `late_slope`: linear slope over the last five available steps; negative is late degradation.

These are designed to quantify the visual degradation in the seed-29 line graph.

In [ ]:
def belief_degradation_table(step_df):
    rows = []
    if step_df.empty:
        return pd.DataFrame()
    for (variant, variant_label, seed), sub in step_df.dropna(subset=['belief_confidence']).groupby(['variant', 'variant_label', 'seed'], observed=True):
        mean_by_step = sub.groupby('step')['belief_confidence'].mean().sort_index()
        if mean_by_step.empty:
            continue
        peak = float(mean_by_step.max())
        peak_step = int(mean_by_step.idxmax())
        final = float(mean_by_step.iloc[-1])
        final_step = int(mean_by_step.index[-1])
        last3 = float(mean_by_step.tail(3).mean())
        tail = mean_by_step.tail(min(5, len(mean_by_step)))
        late_slope = float(np.polyfit(tail.index.to_numpy(dtype=float), tail.to_numpy(dtype=float), 1)[0]) if len(tail) >= 2 else np.nan
        rows.append({
            'variant': variant,
            'variant_label': variant_label,
            'seed': seed,
            'first_step_mean': float(mean_by_step.iloc[0]),
            'peak_confidence': peak,
            'peak_step': peak_step,
            'final_confidence': final,
            'final_step': final_step,
            'peak_to_final_drop': peak - final,
            'late_drop': peak - last3,
            'late_slope': late_slope,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(['variant', 'seed']).reset_index(drop=True)
    return out

degradation_df = belief_degradation_table(step_df)
display(degradation_df)

if not degradation_df.empty:
    mean_degradation = degradation_df.groupby('variant_label', observed=True)[['peak_to_final_drop', 'late_drop', 'late_slope']].mean()
    display(mean_degradation)

    ax = degradation_df.pivot(index='seed', columns='variant_label', values='peak_to_final_drop').plot(kind='bar', figsize=(12, 5))
    ax.set_title('Belief-confidence peak-to-final drop by seed')
    ax.set_ylabel('peak - final confidence')
    ax.axhline(0, color='black', linewidth=1)
    plt.tight_layout()
    plt.show()


## 8. Degradation by Context, Outcome, and Scenario Family

In [ ]:
if step_df.empty:
    print('No step traces loaded.')
else:
    context_conf = step_df.dropna(subset=['belief_confidence']).groupby(
        ['variant_label', 'context', 'step'], observed=True
    )['belief_confidence'].mean().reset_index()
    display(context_conf.head())

    for label in context_conf['variant_label'].dropna().unique():
        sub = context_conf[context_conf['variant_label'] == label]
        pivot = sub.pivot_table(index='context', columns='step', values='belief_confidence', aggfunc='mean')
        if pivot.empty:
            continue
        plt.figure(figsize=(12, max(3, 0.6 * len(pivot))))
        plt.imshow(pivot.values, aspect='auto', vmin=0, vmax=1, cmap='viridis')
        plt.colorbar(label='mean belief confidence')
        plt.yticks(range(len(pivot.index)), pivot.index)
        plt.xticks(range(len(pivot.columns)), pivot.columns)
        plt.xlabel('step')
        plt.title(f'Belief confidence heatmap by context | {label}')
        plt.tight_layout()
        plt.show()

    family_summary = scenario_df.groupby(['variant_label', 'scenario_family', 'outcome'], observed=True).size().rename('n').reset_index()
    display(family_summary)


## 9. REBUS Diagnostics Overview

These summaries are specific to REBUS-family runs. They help diagnose whether a variant is mostly changing alpha, broadening the policy mask, or targeting a repair window.

In [ ]:

REBUS_TRACE_COLS = [
    'rebus_alpha',
    'rebus_prediction_error',
    'rebus_prospective_error',
    'rebus_reflective_error',
    'rebus_repair_window',
    'rebus_surprise',
    'rebus_outcome_surprise',
    'rebus_explicit_error',
    'rebus_resolution_signal',
]

available_trace_cols = [c for c in REBUS_TRACE_COLS if c in step_df.columns]
diag_df = step_df[step_df['variant'] != 'auxhead_clear_reference'].copy()
if diag_df.empty or not available_trace_cols:
    print('No REBUS diagnostics loaded yet.')
else:
    diag_summary = diag_df.groupby(['variant_label', 'seed'], observed=True)[available_trace_cols].agg(['mean', 'max'])
    display(diag_summary)

    mean_diag = diag_df.groupby('variant_label', observed=True)[available_trace_cols].mean()
    display(mean_diag)

    for col in ['rebus_alpha', 'rebus_prediction_error', 'rebus_reflective_error', 'rebus_outcome_surprise', 'rebus_repair_window', 'rebus_explicit_error', 'rebus_resolution_signal']:
        if col not in diag_df or diag_df[col].dropna().empty:
            continue
        summary = summarize_by_step(diag_df, value_col=col)
        if summary.empty:
            continue
        plt.figure(figsize=(12, 5))
        for label, sub in summary.groupby('variant_label', observed=True):
            sub = sub.sort_values('step')
            plt.plot(sub['step'], sub['mean'], marker='o', label=label)
        plt.xlabel('step')
        plt.ylabel(col)
        plt.title(f'{col} by step')
        plt.grid(True, alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()


## 10. Policy-Mask and Repair-Window Diagnostics

This section estimates mask rates from `experiment_mask_firing_counts` where available, and from traces where available. It is intended to catch over-broad interventions like the first reflective run.

In [ ]:
def mask_count_table(analysis_payloads):
    rows = []
    for (variant, seed), payload in analysis_payloads.items():
        counts = payload.get('experiment_mask_firing_counts') or {}
        rates = payload.get('experiment_mask_firing_rates') or {}
        total_steps = sum(len(s.get('action_trace', []) or []) for s in payload.get('scenario_summaries', []))
        mask_names = sorted(set(counts) | set(rates))
        for name in mask_names:
            rows.append({
                'variant': variant,
                'variant_label': VARIANT_LABELS.get(variant, variant),
                'seed': seed,
                'mask': name,
                'count': counts.get(name, np.nan),
                'rate': rates.get(name, counts.get(name, 0) / max(1, total_steps)),
                'total_steps': total_steps,
            })
    return pd.DataFrame(rows)

mask_df = mask_count_table(analysis_payloads)
if mask_df.empty:
    print('No mask counts found yet.')
else:
    display(mask_df.sort_values(['variant', 'seed', 'mask']))
    mask_pivot = mask_df.pivot_table(index='mask', columns='variant_label', values='rate', aggfunc='mean').fillna(0)
    display(mask_pivot)
    plt.figure(figsize=(12, max(4, 0.45 * len(mask_pivot))))
    plt.imshow(mask_pivot.values, aspect='auto', cmap='magma')
    plt.colorbar(label='mean mask rate')
    plt.yticks(range(len(mask_pivot.index)), mask_pivot.index)
    plt.xticks(range(len(mask_pivot.columns)), mask_pivot.columns, rotation=30, ha='right')
    plt.title('Mean mask firing rates by variant')
    plt.tight_layout()
    plt.show()


## 11. Context-Specific Intervention Rates

Trace-derived rates for reflective/hybrid diagnostics. Useful for checking whether repair is concentrated in tight contexts or leaking into normal/high-urgency flow.

In [ ]:

trace_mask_cols = {
    'reflection_active_est': ('rebus_reflective_error', 0.12),
    'repair_window_active_est': ('rebus_repair_window', 0.0),
    'explicit_error_active_est': ('rebus_explicit_error', 0.40),
    'resolution_active_est': ('rebus_resolution_signal', 0.0),
}

if diag_df.empty:
    print('No REBUS trace diagnostics loaded yet.')
else:
    ctx_rows = []
    for (variant_label, seed, context), sub in diag_df.groupby(['variant_label', 'seed', 'context'], observed=True):
        row = {
            'variant_label': variant_label,
            'seed': seed,
            'context': context,
            'steps': len(sub),
            'mean_alpha': sub['rebus_alpha'].mean() if 'rebus_alpha' in sub else np.nan,
            'mean_reflective_error': sub['rebus_reflective_error'].mean() if 'rebus_reflective_error' in sub else np.nan,
            'mean_outcome_surprise': sub['rebus_outcome_surprise'].mean() if 'rebus_outcome_surprise' in sub else np.nan,
            'mean_explicit_error': sub['rebus_explicit_error'].mean() if 'rebus_explicit_error' in sub else np.nan,
            'mean_resolution_signal': sub['rebus_resolution_signal'].mean() if 'rebus_resolution_signal' in sub else np.nan,
        }
        for out_name, (col, threshold) in trace_mask_cols.items():
            if col in sub:
                row[out_name] = (sub[col].fillna(0) > threshold).mean()
        ctx_rows.append(row)
    context_diag_df = pd.DataFrame(ctx_rows)
    display(context_diag_df.sort_values(['variant_label', 'seed', 'context']))

    for metric in ['mean_alpha', 'mean_reflective_error', 'mean_outcome_surprise', 'mean_explicit_error', 'mean_resolution_signal', 'reflection_active_est', 'repair_window_active_est', 'explicit_error_active_est', 'resolution_active_est']:
        if metric not in context_diag_df.columns:
            continue
        pivot = context_diag_df.pivot_table(index='context', columns='variant_label', values=metric, aggfunc='mean')
        if pivot.empty:
            continue
        plt.figure(figsize=(12, max(3, 0.6 * len(pivot))))
        plt.imshow(pivot.values, aspect='auto', cmap='viridis')
        plt.colorbar(label=metric)
        plt.yticks(range(len(pivot.index)), pivot.index)
        plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha='right')
        plt.title(f'{metric} by context and variant')
        plt.tight_layout()
        plt.show()


## 12. Per-Seed Drilldown

Change `SELECTED_SEED` and rerun the cells below to inspect a particular degradation profile.

In [ ]:
SELECTED_SEED = 29

plot_belief_confidence_by_step(seed=SELECTED_SEED)

if not degradation_df.empty:
    display(degradation_df[degradation_df['seed'] == SELECTED_SEED].sort_values('variant'))

if not step_df.empty:
    seed_steps = step_df[step_df['seed'] == SELECTED_SEED]
    display(seed_steps.groupby(['variant_label', 'outcome'], observed=True).size().rename('step_count').reset_index())


## 13. Variant Selection Helper

Use this to focus on one run after all logs are downloaded.

In [ ]:

SELECTED_VARIANT = 'rebus_hybrid_explicit_mask'  # options are keys from EXPECTED_RUN_ROOTS

label = VARIANT_LABELS.get(SELECTED_VARIANT, SELECTED_VARIANT)
print('selected:', label)
print('available variants:', ', '.join(VARIANT_ORDER))
if not metric_df.empty:
    display(metric_df[metric_df['variant'].astype(str) == SELECTED_VARIANT][['variant_label', 'seed'] + METRIC_COLUMNS])
if not mask_df.empty:
    display(mask_df[mask_df['variant'].astype(str) == SELECTED_VARIANT].sort_values(['seed', 'mask']))
plot_belief_confidence_by_step(variants=[SELECTED_VARIANT], include_reference=False)



## 14. Notes for New Runs

This notebook now includes the downloaded explicit-mask run at:

`logs/modal/auxhead-clear-rebus-reflective-hybrid-explicit-mask-20260621/auxhead-clear-rebus-reflective-hybrid-explicit-mask-20260621`

It also includes a placeholder for the resolution run, which will load automatically after it is downloaded to:

`logs/modal/auxhead-clear-rebus-explicit-resolution-20260621/auxhead-clear-rebus-explicit-resolution-20260621`

The most important cells to revisit after adding a run are:
- headline metric table
- metric deltas vs prospective REBUS
- belief-confidence degradation table
- REBUS diagnostics overview
- mask firing rates
- context-specific repair-window, explicit-error, and resolution diagnostics
